# Calora — Train YOLO on Colab GPU\n\n1. Runtime → Change runtime type → **GPU (T4)**\n2. Paste Farq + Calora keys below (read-only for Farq)\n3. Run all cells\n4. Download `best.onnx` + `labels.json` + `nutrition.sqlite` at the end\n

In [ ]:
# === PASTE SECRETS HERE (do not share) ===\nimport os\nos.environ['FARQ_SUPABASE_URL'] = 'https://mpgbvtaguerncgbzvpwg.supabase.co'\nos.environ['FARQ_SUPABASE_SERVICE_KEY'] = 'PASTE_FARQ_SERVICE_KEY'\nos.environ['CALORIE_SUPABASE_URL'] = 'https://ajwsmbysakuukgfewaei.supabase.co'\nos.environ['CALORIE_SUPABASE_SERVICE_KEY'] = 'PASTE_CALORIE_SERVICE_KEY'\nos.environ['FARQ_MAX_ROWS'] = '20000'\nos.environ['MAX_CLASSES'] = '500'\nos.environ['MIN_IMAGES_PER_CLASS'] = '4'\nos.environ['TRAIN_EPOCHS'] = '80'\nos.environ['BATCH_SIZE'] = '16'\nprint('config ok')\n

In [ ]:
!nvidia-smi\n!git clone --depth 1 https://github.com/farq-sa/Get-calo.git /content/Get-calo\n%cd /content/Get-calo\n!git fetch origin cursor/ai-calorie-scanner-929b && git checkout cursor/ai-calorie-scanner-929b || true\n%cd /content/Get-calo/ml\n!pip -q install ultralytics supabase opencv-python-headless pillow imagehash aiohttp aiofiles python-dotenv pyyaml tqdm onnx onnxruntime-gpu httpx pydantic-settings\n

In [ ]:
from pathlib import Path\nenv = '\n'.join(f'{k}={v}' for k,v in os.environ.items() if k.startswith(('FARQ_','CALORIE_','TRAIN_','BATCH_','MAX_','MIN_','YOLO_')))\nPath('/content/Get-calo/.env').write_text(env+'\n')\nPath('/content/Get-calo/ml/.env').write_text(env+'\n')\nprint('env written')\n

In [ ]:
import sys\nsys.path.insert(0, '/content/Get-calo/ml')\nfrom dataset.generate import generate_dataset\nds = generate_dataset(dataset_name='farq_yolo')\nprint(ds)\n

In [ ]:
from train.train_yolo import train_yolo\nfrom pathlib import Path\nrun = train_yolo(\n    Path('data/datasets/farq_yolo/data.yaml'),\n    model='yolov8n.pt',\n    epochs=int(os.environ.get('TRAIN_EPOCHS','80')),\n    batch=int(os.environ.get('BATCH_SIZE','16')),\n    device='0',\n    run_name='farq_colab_v1',\n    workers=2,\n)\nprint(run)\n

In [ ]:
from export.export_models import export_models\nfrom pathlib import Path\nout = export_models(\n    Path('models/runs/farq_colab_v1/weights/best.pt'),\n    labels_json=Path('data/datasets/farq_yolo/labels.json'),\n    out_dir=Path('models/exports/farq_colab_v1'),\n    include_tflite=False,\n    include_coreml=False,\n)\nprint(out)\n!ls -lah models/exports/farq_colab_v1\nfrom google.colab import files\nfor name in ['best.onnx','labels.json','nutrition.sqlite','manifest.json']:\n    p = f'models/exports/farq_colab_v1/{name}'\n    if Path(p).exists():\n        files.download(p)\n